In [1]:
# %matplotlib widget
import sys
sys.path.append('/root/capsule/code/beh_ephys_analysis')
import numpy as np
from pynwb import NWBFile, TimeSeries, NWBHDF5IO
import ast
from pathlib import Path
import glob
from utils.beh_functions import session_dirs, parseSessionID, get_session_tbl, get_unit_tbl, get_history_from_nwb
%matplotlib inline

In [ ]:
# new behavior + ephys
example_session = 'behavior_754897_2025-03-13_11-20-42'

In [4]:
# old behavior + ephys
example_session = 'behavior_ZS061_2021-04-08_18-01-30'

In [3]:
unit_tbl = get_unit_tbl(example_session, data_type='curated')

In [4]:
session_tbl = get_session_tbl(example_session)

In [5]:
from utils.nwb_combining import build_nwb_from_session_and_unit_tables

# Build a fresh NWB from session/unit tables
out_path, nwb_new = build_nwb_from_session_and_unit_tables(
    example_session,
    data_type='curated',
    save_file=None,
    summary=True,
 )

print('New NWB saved to:', out_path)
print('Trials in new NWB:', len(nwb_new.trials))
print('Units in new NWB:', len(nwb_new.units) if nwb_new.units is not None else 0)

TypeError: NWBFile.add_unit: incorrect type for 'waveform_mean' (got 'str', expected 'ndarray, list, tuple, Dataset, Array, StrDataset, HDMFDataset or AbstractDataChunkIterator'), incorrect type for 'waveform_sd' (got 'float', expected 'ndarray, list, tuple, Dataset, Array, StrDataset, HDMFDataset or AbstractDataChunkIterator')

In [ ]:
# Validate trials and units were transferred from tables
session_tbl = get_session_tbl(example_session)
unit_tbl = get_unit_tbl(example_session, data_type='curated')

new_trials_df = nwb_new.trials.to_dataframe()
new_units_df = nwb_new.units.to_dataframe() if nwb_new.units is not None else None

assert len(new_trials_df) == len(session_tbl), (
    f'Trial row mismatch: {len(new_trials_df)} vs {len(session_tbl)}'
 )

missing_trial_cols = [c for c in session_tbl.columns if c not in new_trials_df.columns]
assert not missing_trial_cols, f'Missing trial columns: {missing_trial_cols}'

if unit_tbl is not None and len(unit_tbl) > 0:
    # Core columns should exist in transferred units table
    unit_cols_expected = [c for c in unit_tbl.columns if c not in ('spike_times', 'obs_intervals')]
    missing_unit_cols = [c for c in unit_cols_expected if c not in new_units_df.columns]
    assert not missing_unit_cols, f'Missing unit columns: {missing_unit_cols}'
    assert len(new_units_df) == len(unit_tbl), (
        f'Unit row mismatch: {len(new_units_df)} vs {len(unit_tbl)}'
    )

print('Transfer checks passed.')
print('Trial columns copied:', len(new_trials_df.columns))
print('Unit columns copied:', 0 if new_units_df is None else len(new_units_df.columns))